# Logistic Regression Modeling & Business Recommendations

## 1. Model Training & Evaluation

I built a logistic regression model to predict customer churn, using scikit-learn with standardized features and an 80/20 train-test split.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
df = pd.read_csv('/Users/arkyaghosh/churn_project_data/processed/telco_model_ready.csv')
y = df['Churn']
X = df.drop('Churn', axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

### Initial Results

| Metric | Value |
|--------|-------|
| Accuracy | 78.75% |
| Precision | 62.06% |
| Recall | 51.60% |
| F1-Score | 56.35% |

### Initial Assessment

The baseline model shows solid accuracy, but **recall is the critical metric for churn prediction.** With only 51.6% recall, the model catches just over half of actual churners—which isn't acceptable for an effective retention strategy.

For churn specifically, missing a customer who actually leaves (lost revenue) is far worse than spending retention budget on someone who would have stayed anyway. This means I need to improve recall, even if it means flagging more customers for retention efforts. Therefore, it's important to find a good balance between recall and precision.

## 2 Threshold Optimization

The default logistic regression threshold is 0.5: predict churn if probability ≥ 0.5. This threshold is too conservative for churn prediction. By lowering it to 0.3, I can increase recall while maintaining reasonable precision.

In [ ]:
churn_probs = model.predict_proba(X_test_scaled)[:, 1]
y_pred_new = (churn_probs >= 0.3).astype(int)

accuracy = accuracy_score(y_test, y_pred_new)
precision = precision_score(y_test, y_pred_new)
recall = recall_score(y_test, y_pred_new)
f1 = f1_score(y_test, y_pred_new)
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

### Optimized Results (Threshold = 0.3)

| Metric | Baseline | Optimized | Change |
|--------|----------|-----------|--------|
| Accuracy | 78.75% | 75.41% | -3.3% |
| Precision | 62.06% | 52.57% | -9.5% |
| Recall | 51.60% | 76.47% | +24.9% |
| F1-Score | 56.35% | 62.31% | +6.0% |


### Threshold Rationale

At a 0.3 threshold, I found the best balance between recall and precision. Accuracy decreased slightly, but recall improved significantly—which matters more for our goal. Now the model catches about 76.5% of actual churners while still maintaining reasonable precision at 52.57%, meaning roughly half the customers I target will actually churn.

## 3. Feature Importance

The model learned 30 coefficients—one weight per feature. Positive coefficients push toward churn; negative coefficients push away from churn. I identified the top 5 most influential features in each direction.

In [ ]:
coefficients = model.coef_[0]
sorted_coefficients = coefficients.argsort()
print(f"Features that prevented churn: \n1) {X.columns[sorted_coefficients[0]]} \n2) {X.columns[sorted_coefficients[1]]} \n3) {X.columns[sorted_coefficients[2]]} \n4) {X.columns[sorted_coefficients[3]]} \n5) {X.columns[sorted_coefficients[4]]}")

print(f"\nFeatures that caused churn: \n1) {X.columns[sorted_coefficients[-1]]} \n2) {X.columns[sorted_coefficients[-2]]} \n3) {X.columns[sorted_coefficients[-3]]} \n4) {X.columns[sorted_coefficients[-4]]} \n5) {X.columns[sorted_coefficients[-5]]}")

### Top Features Preventing Churn

| Rank | Feature |
|------|---------|
| 1 | **Tenure** |
| 2 | **Monthly Charges** |
| 3 | **Contract_Two year** |
| 4 | **Contract_One year** |
| 5 | **OnlineSecurity_Yes** |

The general consensus is that the longer someone stays with the company, the less likely they are to churn. Additionally, those with longer contracts and online security are also significantly less likely to churn.

### Top Features Causing Churn

| Rank | Feature |
|------|---------|
| 1 | **TotalCharges** |
| 2 | **InternetService_Fiber optic** |
| 3 | **StreamingMovies_Yes** |
| 4 | **StreamingTV_Yes** |
| 5 | **MultipleLines_Yes** |

The model shows that customers with certain service combinations—particularly streaming services and fiber optic internet—have higher churn probability. However, I cannot determine if it's the quality of these services that causes churn or if customers of these services are simply more price-sensitive. Further investigation would be needed.

## 4. Confounding Variables

A surprising finding emerged: **higher monthly charges predict lower churn** in the model, yet **Week 2 EDA showed churners pay more monthly** on average.

### The Contradiction

One feature that puzzled me was `MonthlyCharges` being among the features that prevented churn. This didn't make sense initially, since Week 2 analysis showed that those paying higher monthly charges were more likely to churn.

The only explanation is that monthly charges is a **confounding variable.** It's likely the combination of high tenure and high monthly charges that results in lower churn rates:

- **High tenure + High monthly charges** = A loyal customer paying for premium services long-term (less likely to churn)
- **Low tenure + High monthly charges** = A newer customer with expensive services who may leave if they find the service expensive (more likely to churn)
This insight is critical: **new customers with high-cost service bundles need early retention focus.**

## 5. Business Recommendations

Based on my model findings, I would recommend the following approach to reduce churn.

### Core Strategy

The model shows that tenure is the most important factor in preventing churn, followed by contract length and online security services. My client should push longer contracts with incentives and bundle them with features that prevent churn, such as online security. This will attract newer customers and help prevent them from churning.

Additionally, I would investigate service quality on features that cause churn. By implementing customer surveys to understand why certain features like fiber optic internet and streaming services are associated with higher churn, the company can determine whether the issue is product quality, pricing, or customer expectations. I would also recommend comparing my client's services and pricing with competitors to see whether they're competitive in the market.

### Implementation Timeline


#### Immediate Actions (0-3 months):
* Offer discounts on 2-year contracts bundled with online security
* Begin customer surveys for fiber optic service feedback

#### Medium-term (3-6 months):
* Complete competitive benchmarking analysis
* Develop service quality improvements based on survey findings

#### Long-term (6+ months):
* Roll out improved services and adjust pricing if needed
* Monitor churn rates to measure effectiveness

---


## Summary

The logistic regression model identified **tenure and contract length as the dominant churn drivers**, with **online security as a powerful protective factor.** By optimizing the decision threshold to prioritize recall, I can now catch approximately 76% of churners. The company should focus on promoting longer contracts with bundled online security, investigating quality issues with fiber optic and streaming services, and providing early retention support for new customers—especially those with high-cost service bundles.

